# Practical 4 – Optimizer Comparison
**Project:** NoteReader – Indian Currency Speaker for the Blind  
**Institute:** RCOEM, Nagpur – CSE Deep Learning Lab  
**Mapped to:** P4 (Optimization Algorithms)  

## Objectives
1. Train the P6 CNN baseline with 6 different optimizers: SGD, Momentum SGD, Nesterov, Adagrad, RMSProp, Adam.
2. Compare convergence speed, final accuracy and training stability across batch sizes 16, 32, 64.
3. Understand *why* Adam outperforms SGD on this task (adaptive learning rates + momentum).
4. Optional NumPy implementations of vanilla GD, Momentum and Adam on a toy loss surface.

## Viva Q&A

### Q – Which optimizer converges fastest on this dataset?
**Adam** typically converges fastest (fewest epochs to reach ≥80 % val accuracy) because it combines  
adaptive per-parameter learning rates (like Adagrad/RMSProp) with first-order momentum (like Momentum SGD).  
It maintains exponential moving averages of both gradients (m) and squared gradients (v),  
and corrects for their initialisation bias:
```
m_t = β₁·m_{t-1} + (1-β₁)·g_t           # 1st moment (mean)
v_t = β₂·v_{t-1} + (1-β₂)·g_t²          # 2nd moment (variance)
m̂_t = m_t / (1-β₁ᵗ)                    # bias-corrected
v̂_t = v_t / (1-β₂ᵗ)
θ_t = θ_{t-1} - α · m̂_t / (√v̂_t + ε)
```

### Q – Which is most stable?
**RMSProp** and **Adam** are both stable; plain SGD with large LR can oscillate.

### Q – Why does Adam differ from SGD with Momentum?
Momentum SGD uses a **global** learning rate for all parameters.  
Adam uses **per-parameter** adaptive rates scaled by the historical squared gradient —  
sparse parameters (uncommon features) get larger updates, dense parameters get smaller ones.  
This is crucial for convolutional filters that see different patterns.

### Q – Why does batch size matter?
- **Small batch (16)**: noisier gradient → acts as regulariser; slower per-epoch wall-clock but often better generalisation.
- **Large batch (64)**: smoother gradient → faster per-step; can overfit and land in sharp minima.
- **32** is the classic compromise for small datasets like ours.

In [ ]:
"""
Setup – load the saved dataset splits (same as P6).
We MUST use the same split to make comparisons valid.
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json, os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# Load the splits saved in P6
X_train = np.load('../dataset/X_train.npy')
y_train = np.load('../dataset/y_train.npy')
X_val   = np.load('../dataset/X_val.npy')
y_val   = np.load('../dataset/y_val.npy')
X_test  = np.load('../dataset/X_test.npy')
y_test  = np.load('../dataset/y_test.npy')

CLASSES = ['10', '20', '50', '100', '200', '500']
NUM_CLASSES = 6
EPOCHS_OPT  = 20   # keep short for comparison

# Normalise [0,1]
X_train_n = X_train / 255.0
X_val_n   = X_val   / 255.0
X_test_n  = X_test  / 255.0

y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

# Same augmentation pipeline as P6
datagen = ImageDataGenerator(
    rotation_range=30, zoom_range=0.2, brightness_range=[0.6, 1.4],
    width_shift_range=0.15, height_shift_range=0.15,
    horizontal_flip=True, shear_range=10, fill_mode='nearest')
datagen.fit(X_train_n)
print("Splits loaded. Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

In [ ]:
"""
Helper function – re-build a fresh CNN (same architecture as P6 baseline).
We rebuild it every run to ensure fair comparison (same initial weights via SEED).
"""

def build_cnn(seed=SEED):
    """Fresh CNN scratch model with fixed random seed for fair comparison."""
    tf.random.set_seed(seed)
    model = Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same',
                      input_shape=(224,224,3)),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128,(3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    return model


# ── Define all 6 optimizers with their standard hyperparameters ───────────────
# We keep LR=0.01 for SGD-family (common default) and 0.001 for adaptive methods.
OPTIMIZERS = {
    'SGD':      keras.optimizers.SGD(learning_rate=0.01),
    'Momentum': keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'Nesterov': keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    'Adagrad':  keras.optimizers.Adagrad(learning_rate=0.01),
    'RMSProp':  keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam':     keras.optimizers.Adam(learning_rate=0.001),
}

BATCH_SIZE = 32   # default; we vary this later

histories   = {}
test_accs   = {}

os.makedirs('../results', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

for name, opt in OPTIMIZERS.items():
    print(f"\n{'='*50}")
    print(f" Training with optimizer: {name}")
    print(f"{'='*50}")
    model = build_cnn()
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

    hist = model.fit(
        datagen.flow(X_train_n, y_train_oh, batch_size=BATCH_SIZE, seed=SEED),
        epochs=EPOCHS_OPT,
        validation_data=(X_val_n, y_val_oh),
        verbose=0   # suppress per-batch output; final epoch printed below
    )
    _, test_acc = model.evaluate(X_test_n, y_test_oh, verbose=0)
    test_accs[name] = round(test_acc * 100, 2)
    histories[name] = hist.history
    print(f"  → {name}: Test Acc = {test_acc*100:.2f}%")

print("\nAll optimizers trained.")

In [ ]:
"""
Plot: Val accuracy and Val loss across all optimizers on the same graph.
This is the KEY comparison chart for the viva.
"""

COLOR_MAP = {
    'SGD': 'gray', 'Momentum': 'blue', 'Nesterov': 'cyan',
    'Adagrad': 'orange', 'RMSProp': 'green', 'Adam': 'red'
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

for name, hist in histories.items():
    ax1.plot(hist['val_loss'],     label=name, color=COLOR_MAP[name])
    ax2.plot(hist['val_accuracy'], label=name, color=COLOR_MAP[name])

ax1.set(title='Val Loss – All Optimizers', xlabel='Epoch', ylabel='Loss')
ax1.legend()
ax2.set(title='Val Accuracy – All Optimizers', xlabel='Epoch', ylabel='Accuracy')
ax2.legend()
plt.suptitle('P4 – Optimizer Comparison (Batch=32)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/plots/optimizer_comparison_bs32.png', dpi=150)
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
df_opt = pd.DataFrame({
    'Optimizer': list(test_accs.keys()),
    'Test Accuracy (%)': list(test_accs.values()),
    'Final Val Loss': [round(histories[n]['val_loss'][-1], 4) for n in test_accs]
})
df_opt.sort_values('Test Accuracy (%)', ascending=False, inplace=True)
print("\nOptimizer Comparison Summary (batch=32):")
print(df_opt.to_string(index=False))
df_opt.to_csv('../results/optimizer_comparison.csv', index=False)

In [ ]:
"""
Batch-size experiment: compare batch sizes 16, 32, 64 with Adam (best optimizer).
"""

batch_histories = {}
for bs in [16, 32, 64]:
    print(f"\n── Adam, Batch size = {bs} ──")
    model = build_cnn()
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(
        datagen.flow(X_train_n, y_train_oh, batch_size=bs, seed=SEED),
        epochs=EPOCHS_OPT,
        validation_data=(X_val_n, y_val_oh),
        verbose=0)
    _, test_acc = model.evaluate(X_test_n, y_test_oh, verbose=0)
    batch_histories[f'bs={bs}'] = hist.history
    print(f"  Test Acc = {test_acc*100:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for label, hist in batch_histories.items():
    ax1.plot(hist['val_loss'],     label=label)
    ax2.plot(hist['val_accuracy'], label=label)
ax1.set(title='Val Loss – Batch Size', xlabel='Epoch', ylabel='Loss'); ax1.legend()
ax2.set(title='Val Acc  – Batch Size', xlabel='Epoch', ylabel='Acc');  ax2.legend()
plt.suptitle('P4 – Batch Size Comparison (Adam)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/plots/batch_size_comparison.png', dpi=150)
plt.show()

## Optional – NumPy from-scratch implementations of GD, Momentum, Adam
These mirror what a first-principles DL student is expected to implement in Practical 4.

In [ ]:
"""
Toy example: minimise f(x) = x² using vanilla GD, Momentum, and Adam.
Plots the trajectory of x to the minimum (x=0) for each optimizer.
This is purely to understand and explain the algorithm at the viva.
"""

def grad_f(x): return 2 * x   # gradient of f(x)=x²

def gd(start=10.0, lr=0.1, steps=50):
    x, traj = start, [start]
    for _ in range(steps):
        x -= lr * grad_f(x)
        traj.append(x)
    return traj

def momentum_gd(start=10.0, lr=0.1, beta=0.9, steps=50):
    x, v, traj = start, 0.0, [start]
    for _ in range(steps):
        v = beta * v + (1 - beta) * grad_f(x)  # velocity
        x -= lr * v
        traj.append(x)
    return traj

def adam_np(start=10.0, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, steps=50):
    x, m, v, traj = start, 0.0, 0.0, [start]
    for t in range(1, steps + 1):
        g = grad_f(x)
        m = beta1 * m + (1 - beta1) * g            # 1st moment
        v = beta2 * v + (1 - beta2) * g**2         # 2nd moment
        m_hat = m / (1 - beta1**t)                 # bias correction
        v_hat = v / (1 - beta2**t)
        x -= lr * m_hat / (np.sqrt(v_hat) + eps)   # parameter update
        traj.append(x)
    return traj

steps = 40
t_gd  = gd(steps=steps)
t_mom = momentum_gd(steps=steps)
t_adam= adam_np(steps=steps)

plt.figure(figsize=(10, 4))
plt.plot(t_gd,   label='Vanilla GD',  color='gray')
plt.plot(t_mom,  label='Momentum GD', color='blue')
plt.plot(t_adam, label='Adam',        color='red')
plt.axhline(0, color='black', linestyle='--', linewidth=0.7, label='Optimum (x=0)')
plt.xlabel('Step'); plt.ylabel('x')
plt.title('NumPy Optimizer Trajectories on f(x) = x²')
plt.legend(); plt.tight_layout()
plt.savefig('../results/plots/numpy_optimizer_trajectories.png', dpi=150)
plt.show()
print(f"GD final x:       {t_gd[-1]:.6f}")
print(f"Momentum final x: {t_mom[-1]:.6f}")
print(f"Adam final x:     {t_adam[-1]:.6f}")